In [1]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,000 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease                         
Get:7 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]      
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]           
Get:9 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]       
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]  
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]     
Ge

In [2]:
!nohup ollama serve > ollama.log 2>&1 &
!sleep 3 && tail -2 ollama.log

time=2026-11-03T00:31:45.555Z level=INFO source=runner.go:60 msg="discovering available GPUs..."
time=2026-11-03T00:31:45.820Z level=INFO source=model_recommendations.go:177 msg="model recommendations cache sleep scheduled" wait=3h52m52.371863991s consecutive_failures=0


In [3]:
!ollama pull qwen2.5:7b

In [4]:
from openai import OpenAI
import requests
print(requests.get("http://127.0.0.1:11434").content)

b'Ollama is running'


In [5]:
ollama_url = 'http://127.0.0.1:11434/v1'
ollama = OpenAI(base_url=ollama_url, api_key='ollama')

In [6]:
import pandas as pd
df_processed_btx_BR = pd.read_csv("finBer_classification_Bullish.csv").reset_index(drop=True)
#df_processed_btx_BM = pd.read_csv("classification_Bearmarket.csv").reset_index(drop=True)


In [ ]:
#!curl http://127.0.0.1:11434/api/tags

In [10]:
!pkill ollama 2>/dev/null
!OLLAMA_NUM_PARALLEL=8 nohup ollama serve > ollama.log 2>&1 &
!sleep 3 && curl -s http://127.0.0.1:11434/api/tags

{"models":[{"name":"qwen2.5:7b","model":"qwen2.5:7b","modified_at":"2026-11-03T00:32:42.052362388Z","size":4683087332,"digest":"845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":3584,"runner":"ggml"},"capabilities":["completion","tools"]}]}

In [7]:
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from google.colab import files



def classify(tweet: str) -> str:
    resp = ollama.chat.completions.create(
        model="qwen2.5:7b",
        messages=[{"role": "user", "content": f"classify this message as a bullish, bearish or neutral, when the classification is done sumarize your answer in ONLY one word from: 'bearish', 'bullish' or 'neutral': {tweet}"}],
        temperature=0
    )
    return resp.choices[0].message.content.lower()


In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

SAVE_DIR = os.path.join(os.getcwd(), "Nlp")  # dossier local
os.makedirs(SAVE_DIR, exist_ok=True)

CHUNK_SIZE = 5000
messages = df_processed_btx_BM['message_body'].tolist()
results = []

with ThreadPoolExecutor(max_workers=8) as pool:
    for i in range(0, len(messages), CHUNK_SIZE):
        chunk = messages[i:i + CHUNK_SIZE]
        chunk_results = list(tqdm(pool.map(classify, chunk), total=len(chunk)))
        results.extend(chunk_results)

        # Sauvegarde périodique
        df_processed_btx_BM['sentiment_from_llm'] = results + [None] * (len(messages) - len(results))
        df_processed_btx_BM.to_csv(os.path.join(SAVE_DIR, "processed_btx_BM_llm.csv"), index=False)
        print(f"✓ Sauvegardé {len(results)}/{len(messages)}")

# Sauvegarde finale
df_processed_btx_BM['sentiment_from_llm'] = results
df_processed_btx_BM.to_csv(os.path.join(SAVE_DIR, "classification_bearmarket.csv"), index=False)
print(f"✓ Fichier final : {os.path.join(SAVE_DIR, 'classification_bearmarket.csv')}")   

100%|██████████| 5000/5000 [08:18<00:00, 10.03it/s]


✓ Sauvegardé 5000/136574


100%|██████████| 5000/5000 [08:08<00:00, 10.24it/s]


✓ Sauvegardé 10000/136574


100%|██████████| 5000/5000 [08:21<00:00,  9.97it/s]


✓ Sauvegardé 15000/136574


... (13 outputs left out)

✓ Sauvegardé 50000/136574


100%|██████████| 5000/5000 [07:53<00:00, 10.55it/s]


✓ Sauvegardé 55000/136574


 70%|███████   | 3501/5000 [05:33<02:27, 10.13it/s]

## Nouvelle itération en commençant à parti de la derniere itération


In [8]:
import pandas as pd
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

SAVE_DIR = os.path.join(os.getcwd(), "Nlp")
os.makedirs(SAVE_DIR, exist_ok=True)
PARTIAL_CSV = os.path.join(SAVE_DIR, "classification_bearmarket_partial.csv")
CHUNK_SIZE = 5000

# Charger le DataFrame (avec les colonnes déjà présentes)
df_processed_btx_BM = pd.read_csv("classification_bearmarket_partial.csv")
messages = df_processed_btx_BM['message_body'].tolist()

# Trouver le premier index sans classification
if 'sentiment_from_llm' in df_processed_btx_BM.columns:
    start_idx = df_processed_btx_BM['sentiment_from_llm'].isna().idxmax()
    # Si TOUT est déjà classifié, idxmax() retourne 0 → on gère le cas :
    if not df_processed_btx_BM['sentiment_from_llm'].isna().any():
        print("Tout est déjà classifié ✅")
        start_idx = len(messages)
else:
    start_idx = 0

print(f"Démarrage à l'index {start_idx} / {len(messages)}")

# Pré-remplir results avec ce qui a déjà été fait
results = df_processed_btx_BM['sentiment_from_llm'].tolist() if 'sentiment_from_llm' in df_processed_btx_BM.columns else [None] * len(messages)

# Boucle : on commence là où ça s'est arrêté
with ThreadPoolExecutor(max_workers=8) as pool:
    for i in range(start_idx, len(messages), CHUNK_SIZE):
        chunk = messages[i:i + CHUNK_SIZE]
        chunk_results = list(tqdm(pool.map(classify, chunk), total=len(chunk)))
        results[i:i + len(chunk)] = chunk_results

        # Sauvegarde périodique
        df_processed_btx_BM['sentiment_from_llm'] = results + [None] * (len(messages) - len(results))
        df_processed_btx_BM.to_csv(PARTIAL_CSV)
        print(f"  → {len(results)}/{len(messages)} classifiés")

# Sauvegarde finale
df_processed_btx_BM['sentiment_from_llm'] = results
df_processed_btx_BM.to_csv(os.path.join(SAVE_DIR, "classification_bearmarket.csv"))   

/tmp/ipykernel_741/2618502366.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_processed_btx_BM = pd.read_csv("classification_bearmarket_partial.csv")


Démarrage à l'index 105000 / 136574


100%|██████████| 5000/5000 [10:08<00:00,  8.21it/s]  


  → 136574/136574 classifiés


 17%|█▋        | 842/5000 [01:28<07:17,  9.51it/s]


KeyboardInterrupt: 

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import pandas as pd

# Sauvegarde périodique toutes les 5000 itérations
CHUNK_SIZE = 5000
messages = df_processed_btx_BR['message_body'].tolist()
results = []

with ThreadPoolExecutor(max_workers=8) as pool:
    for i in range(0, len(messages), CHUNK_SIZE):
        chunk = messages[i:i + CHUNK_SIZE]
        chunk_results = list(tqdm(pool.map(classify, chunk), total=len(chunk)))
        results.extend(chunk_results)

        # Sauvegarde locale
        df_processed_btx_BR['sentiment_from_llm'] = results + [None] * (len(messages) - len(results))
        df_processed_btx_BR.to_csv("classification_bullrun_.csv", index=False)

        # Sauvegarde Google Drive
        os.makedirs('/content/drive/My Drive/Nlp', exist_ok=True)
        df_processed_btx_BR.to_csv('/content/drive/My Drive/Nlp/processed_btx_BM_llm.csv', index=False)

        print(f"✓ Sauvegardé {len(results)}/{len(messages)}")

# Sauvegarde finale
df_processed_btx_BM['sentiment_from_llm'] = results
df_processed_btx_BM.to_csv("classification_bearmarket.csv", index=False)
os.makedirs('/content/drive/My Drive/Nlp', exist_ok=True)
df_processed_btx_BM.to_csv('/content/drive/My Drive/Nlp/processed_btx_BM_llm.csv', index=False)
files.download('classification_bearmarket.csv')   

100%|██████████| 5000/5000 [08:25<00:00,  9.89it/s]


✓ Sauvegardé 5000/64789


100%|██████████| 5000/5000 [08:43<00:00,  9.55it/s]


✓ Sauvegardé 10000/64789


100%|██████████| 5000/5000 [08:35<00:00,  9.69it/s]


✓ Sauvegardé 15000/64789


... (11 outputs left out)

✓ Sauvegardé 45000/64789


  7%|▋         | 336/5000 [00:36<07:04, 10.98it/s]

  7%|▋         | 341/5000 [00:36<08:46,  8.85it/s]

: 

In [ ]:
!pip install transformers datasets accelerate scikit-learn pandas -q

In [2]:
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

In [32]:
df = pd.read_csv("classification_bearmarket_partial.csv")
number_ref = 4000
# Sélection équilibrée : 4000 par classe
bullish = df[df["sentiment_from_llm"] == "bullish"].sample(n=number_ref, random_state=50)
bearish = df[df["sentiment_from_llm"] == "bearish"].sample(n=number_ref, random_state=20)
neutral = df[df["sentiment_from_llm"] == "neutral"].sample(n=number_ref, random_state=30)

df_balanced = pd.concat([bearish, bullish, neutral], ignore_index=True)

# Mapping des labels : bearish=0, bullish=1, neutral=2
label_map = {"bearish": 0, "bullish": 1, "neutral": 2}
df_balanced["label"] = df_balanced["sentiment_from_llm"].map(label_map)

# Ajuste le nom de la colonne texte si besoin
# texts_col = "text"  # ← change si ta colonne s'appelle autrement (ex: "tweet", "content")
texts_col = "message_body"

texts = df_balanced[texts_col].tolist()
labels = df_balanced["label"].tolist()
# Split train / val (85/15)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)

print(f"Train: {len(train_texts)} | Val: {len(val_labels)}")
print(f"Répartition train: bearish={train_labels.count(0)}, bullish={train_labels.count(1)}, neutral={train_labels.count(2)}")


Train: 10200 | Val: 1800
Répartition train: bearish=3400, bullish=3400, neutral=3400


/tmp/ipykernel_47830/1788033988.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("classification_bearmarket_partial.csv")


In [33]:
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(
    tokenize, batched=True, desc="Tokenize train"
)
val_ds = Dataset.from_dict({"text": val_texts, "label": val_labels}).map(
    tokenize, batched=True, desc="Tokenize val"
)

Tokenize train:   0%|          | 0/10200 [00:00<?, ? examples/s]

Tokenize val:   0%|          | 0/1800 [00:00<?, ? examples/s]

In [34]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

model.config.id2label = {0: "bearish", 1: "bullish", 2: "neutral"}
model.config.label2id = {"bearish": 0, "bullish": 1, "neutral": 2}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [35]:
from transformers import DataCollatorWithPadding  # ← ajouter en haut de la cellule

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  # ← ajouter avant le Trainer

training_args = TrainingArguments(
    output_dir="./finbert_tweet_classifier",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=100,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,

)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.956231,0.883511,0.588333,0.589741
2,0.769668,0.794677,0.676111,0.667742
3,0.544911,0.702104,0.712778,0.712778
4,0.362746,0.749314,0.713889,0.713642


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1276, training_loss=0.7229877893453855, metrics={'train_runtime': 289.1347, 'train_samples_per_second': 141.111, 'train_steps_per_second': 4.413, 'total_flos': 1505707644420000.0, 'train_loss': 0.7229877893453855, 'epoch': 4.0})

In [36]:
results = trainer.evaluate()
print(f"\nF1 (weighted): {results['eval_f1']:.4f}")
print(f"Accuracy:      {results['eval_accuracy']:.4f}")


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.362746,0.749314,4,0.713889,0.713642



F1 (weighted): 0.7136
Accuracy:      0.7139


In [ ]:
# 1. Fine-tuner (comme avant, epoch 3)
# a cause des difficulté à sauvegarder mon transformer fine sur ma machine locale depuis colab,
#il faut le pusher directement dès qu'il es pret sur HF
# 2. IMMÉDIATEMENT après, pousser sur Hugging Face Hub

from huggingface_hub import HfApi
import os

# a faire une seule fois
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"]
model.push_to_hub("mawuenar/finbert-tweets")
tokenizer.push_to_hub("mawuenar/finbert-tweets")
trainer.save_model("./finbert_tweet_classifier_btx")
tokenizer.save_pretrained("./finbert_tweet_classifier_btx")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...r0kt1mg/model.safetensors:   0%|          | 14.2kB /  438MB            

Validating                    :           |   0%            

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./finbert_tweet_classifier_btx/tokenizer_config.json',
 './finbert_tweet_classifier_btx/tokenizer.json')